# TC-Former Improved — повний пайплайн

Dataset: IP / PU / WHHH. Змінюй `DATASET` і `SEEDS` у наступній клітинці.

In [1]:
# ── Конфігурація запуску ──────────────────────────────────────────────────────
DATASET    = 'IP'          # 'IP' | 'PU' | 'WHHH'
SEEDS      = [42]          # список seeds, напр. [0, 1, 2, 3, 4]
DATA_PATH  = 'data'
RESULTS    = 'results'
PAPER_MODE = False         # True → без val-спліту, test використовується як eval
LOG_EVERY  = 5             # виводити лог кожні N епох

## Імпорти та залежності

In [2]:
import os, json, dataclasses, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Optional
warnings.filterwarnings('ignore')

import numpy as np
import scipy.io
import torch
import torch.nn as nn
import lightning as L
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.metrics import cohen_kappa_score
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger

print('PyTorch:', torch.__version__)
print('Lightning:', L.__version__)

PyTorch: 2.5.1
Lightning: 2.6.1


## Config

In [3]:
@dataclass
class Config:
    dataset: str
    patch_size: int
    pca_components: int
    kernel_size: int
    num_classes: int
    batch_size: int

    data_path: str = 'data'
    hidden_dim: int = 64
    num_heads: int = 1
    dropout: float = 0.1
    depth: int = 2
    lr: float = 5e-4
    epochs: int = 100
    weight_decay: float = 1e-4
    patience: int = 20
    num_train_per_class: int = 10
    num_val_per_class: int = 5
    seed: int = 42
    use_fps: bool = True
    remove_pca_outliers: bool = True
    outlier_std: float = 2.5
    use_augmentation: bool = True
    use_bidirectional_wkv: bool = True
    use_pos_encoding: bool = True
    use_se_block: bool = True


CONFIGS = {
    'IP':   Config(dataset='IP',   patch_size=15, pca_components=150, kernel_size=9,  num_classes=16, batch_size=200),
    'PU':   Config(dataset='PU',   patch_size=25, pca_components=20,  kernel_size=17, num_classes=9,  batch_size=100),
    'WHHH': Config(dataset='WHHH', patch_size=31, pca_components=135, kernel_size=19, num_classes=22, batch_size=20),
}

## Препроцесинг: завантаження, PCA, видалення аутлаєрів

In [4]:
def load_dataset(name: str, data_path: str = 'data'):
    if name == 'IP':
        d = scipy.io.loadmat(os.path.join(data_path, 'IP', 'Indian_pines_corrected.mat'))
        hsi = d['indian_pines_corrected']
        labels = np.load(os.path.join(data_path, 'IP', 'IPgt.npy'))
    elif name == 'PU':
        d = scipy.io.loadmat(os.path.join(data_path, 'Pavia', 'PaviaU.mat'))
        g = scipy.io.loadmat(os.path.join(data_path, 'Pavia', 'PaviaU_gt.mat'))
        hsi, labels = d['paviaU'], g['paviaU_gt']
    elif name == 'WHHH':
        d = scipy.io.loadmat(os.path.join(data_path, 'WHU-Hi-HongHu', 'WHU_Hi_HongHu.mat'))
        g = scipy.io.loadmat(os.path.join(data_path, 'WHU-Hi-HongHu', 'WHU_Hi_HongHu_gt.mat'))
        hsi, labels = d['WHU_Hi_HongHu'], g['WHU_Hi_HongHu_gt']
    else:
        raise ValueError(f'Unknown dataset: {name!r}')
    hsi    = hsi.astype(np.float32)
    labels = labels.astype(np.int64)
    print(f'[{name}] HSI {hsi.shape}  Labels {labels.shape}  Labeled px: {(labels > 0).sum()}')
    return hsi, labels


def apply_pca(hsi: np.ndarray, n_components: int):
    H, W, C = hsi.shape
    flat = hsi.reshape(-1, C)
    flat = (flat - flat.mean(0)) / (flat.std(0) + 1e-8)
    pca  = PCA(n_components=n_components, svd_solver='randomized', random_state=42)
    out  = pca.fit_transform(flat).reshape(H, W, n_components).astype(np.float32)
    print(f'PCA {C} -> {n_components}  explained variance: {pca.explained_variance_ratio_.sum():.3f}')
    return out, pca


def pca_class_outlier_mask(vectors: np.ndarray, outlier_std: float = 2.5) -> np.ndarray:
    centroid  = vectors.mean(axis=0)
    d         = np.linalg.norm(vectors - centroid, axis=1)
    threshold = d.mean() + outlier_std * d.std()
    return d > threshold


def remove_pca_outliers(labels: np.ndarray, hsi_pca: np.ndarray, outlier_std: float = 2.5) -> np.ndarray:
    labels_out      = labels.copy()
    n_labeled_before = int((labels > 0).sum())
    n_removed        = 0
    for cls in range(1, int(labels.max()) + 1):
        pos = np.argwhere(labels == cls)
        if len(pos) == 0:
            continue
        vectors   = hsi_pca[pos[:, 0], pos[:, 1]]
        out_local = pca_class_outlier_mask(vectors, outlier_std)
        if out_local.any():
            out_pos = pos[out_local]
            labels_out[out_pos[:, 0], out_pos[:, 1]] = 0
            n_removed += int(out_local.sum())
    n_after = int((labels_out > 0).sum())
    pct     = 100 * n_removed / n_labeled_before if n_labeled_before else 0.0
    print(f'Outlier removal (>{outlier_std}σ): {n_removed} px removed ({pct:.1f}%)  |  labeled: {n_labeled_before} -> {n_after}')
    return labels_out

## Dataset: сплiт, аугментації, DataLoader

In [5]:
def _fps_indices(vectors: np.ndarray, k: int, outlier_std: float = 2.5) -> np.ndarray:
    n = len(vectors)
    if n <= k:
        return np.arange(n)
    inlier_idx = np.where(~pca_class_outlier_mask(vectors, outlier_std))[0]
    if len(inlier_idx) < k:
        inlier_idx = np.arange(n)
    vecs      = vectors[inlier_idx]
    centroid  = vectors.mean(axis=0)
    d_to_c    = np.linalg.norm(vecs - centroid, axis=1)
    selected  = [int(np.argmin(d_to_c))]
    min_dists = np.full(len(vecs), np.inf)
    for _ in range(k - 1):
        last_vec  = vecs[selected[-1]]
        d_new     = np.linalg.norm(vecs - last_vec, axis=1)
        min_dists = np.minimum(min_dists, d_new)
        selected.append(int(np.argmax(min_dists)))
    return inlier_idx[selected]


def create_split(labels, hsi_pca, n_train_per_class, n_val_per_class=5,
                 seed=42, use_fps=True, outlier_std=2.5):
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []
    for cls in range(1, labels.max() + 1):
        pos  = np.argwhere(labels == cls)
        if len(pos) == 0:
            continue
        n_tr = min(n_train_per_class, len(pos))
        if use_fps and n_tr < len(pos):
            vectors   = hsi_pca[pos[:, 0], pos[:, 1]]
            sel_local = _fps_indices(vectors, n_tr, outlier_std)
            train_pos = pos[sel_local]
            mask      = np.ones(len(pos), dtype=bool)
            mask[sel_local] = False
            remaining_pos = pos[mask]
        else:
            perm          = rng.permutation(len(pos))
            train_pos     = pos[perm[:n_tr]]
            remaining_pos = pos[perm[n_tr:]]
        remaining_pos = remaining_pos[rng.permutation(len(remaining_pos))]
        n_val         = min(n_val_per_class, len(remaining_pos))
        train_idx.extend(train_pos.tolist())
        val_idx.extend(remaining_pos[:n_val].tolist())
        test_idx.extend(remaining_pos[n_val:].tolist())
    return np.array(train_idx), np.array(val_idx), np.array(test_idx)


class SpectralJitter:
    def __init__(self, scale=0.1): self.scale = scale
    def __call__(self, x):
        return x * (1.0 + (torch.rand(x.shape[0], 1, 1) * 2 - 1) * self.scale)

class BandDropout:
    def __init__(self, p=0.15): self.p = p
    def __call__(self, x):
        return x * (torch.rand(x.shape[0], 1, 1) > self.p).float()

class SpatialFlip:
    def __call__(self, x):
        if torch.rand(1).item() > 0.5: x = x.flip(2)
        if torch.rand(1).item() > 0.5: x = x.flip(1)
        return x

class PatchRotation:
    def __call__(self, x):
        return torch.rot90(x, k=torch.randint(0, 4, (1,)).item(), dims=[1, 2])

class ComposeAugmentations:
    def __init__(self, transforms): self.transforms = transforms
    def __call__(self, x):
        for t in self.transforms: x = t(x)
        return x


class HSIPatchDataset(Dataset):
    def __init__(self, hsi_pca, labels, indices, patch_size, augment=None):
        pad          = patch_size // 2
        self.labels  = labels
        self.indices = indices
        self.pad     = pad
        self.hsi     = np.pad(hsi_pca, ((pad, pad), (pad, pad), (0, 0)), mode='reflect')
        self.augment = augment

    def __len__(self): return len(self.indices)

    def __getitem__(self, idx):
        r, c   = self.indices[idx]
        label  = int(self.labels[r, c]) - 1
        rp, cp = r + self.pad, c + self.pad
        patch  = self.hsi[rp - self.pad:rp + self.pad + 1,
                          cp - self.pad:cp + self.pad + 1, :]
        patch  = torch.from_numpy(patch.copy()).permute(2, 0, 1).float()
        if self.augment is not None:
            patch = self.augment(patch)
        return patch, torch.tensor(label, dtype=torch.long)


def get_dataloaders(hsi_pca, labels, cfg):
    train_idx, val_idx, test_idx = create_split(
        labels, hsi_pca,
        n_train_per_class=cfg.num_train_per_class,
        n_val_per_class=cfg.num_val_per_class,
        seed=cfg.seed,
        use_fps=cfg.use_fps,
        outlier_std=cfg.outlier_std,
    )
    print(f'Split — Train: {len(train_idx)}  Val: {len(val_idx)}  Test: {len(test_idx)}')
    train_aug = None
    if cfg.use_augmentation:
        train_aug = ComposeAugmentations([SpectralJitter(), BandDropout(), SpatialFlip(), PatchRotation()])
    train_ds = HSIPatchDataset(hsi_pca, labels, train_idx, cfg.patch_size, augment=train_aug)
    val_ds   = HSIPatchDataset(hsi_pca, labels, val_idx,   cfg.patch_size)
    test_ds  = HSIPatchDataset(hsi_pca, labels, test_idx,  cfg.patch_size)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, shuffle=False, num_workers=0)
    return train_loader, val_loader, test_loader

## Модель: TC-Former Improved

In [6]:
class WKVOperator(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.w_log = nn.Parameter(torch.zeros(dim))
        self.u     = nn.Parameter(torch.zeros(dim))

    def forward(self, k, v):
        B, T, C = k.shape
        w       = -torch.exp(self.w_log)
        idx     = torch.arange(T, device=k.device, dtype=k.dtype)
        exp     = (idx.unsqueeze(1) - idx.unsqueeze(0) - 1).unsqueeze(-1) * w
        causal  = idx.unsqueeze(1) > idx.unsqueeze(0)
        exp_s   = exp.masked_fill(~causal.unsqueeze(-1).expand_as(exp), float('-inf'))
        decay   = torch.exp(exp_s)
        ek      = torch.exp(k)
        dp      = decay.permute(2, 0, 1)
        A       = torch.bmm(dp, (ek * v).permute(2, 1, 0)).permute(2, 1, 0)
        B_den   = torch.bmm(dp, ek.permute(2, 1, 0)).permute(2, 1, 0)
        euk     = torch.exp(self.u + k)
        return (A + euk * v) / (B_den + euk + 1e-8)


class BidirectionalWKVOperator(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.wkv_fwd = WKVOperator(dim)
        self.wkv_bwd = WKVOperator(dim)
        self.merge   = nn.Linear(2 * dim, dim, bias=False)
        with torch.no_grad():
            self.merge.weight.copy_(0.5 * torch.eye(dim).repeat(1, 2))

    def forward(self, k, v):
        fwd = self.wkv_fwd(k, v)
        bwd = self.wkv_bwd(k.flip(1), v.flip(1)).flip(1)
        return self.merge(torch.cat([fwd, bwd], dim=-1))


class TimeMixing(nn.Module):
    def __init__(self, dim, wkv_cls=WKVOperator):
        super().__init__()
        self.mu_r = nn.Parameter(torch.full((1, 1, dim), 0.5))
        self.mu_k = nn.Parameter(torch.full((1, 1, dim), 0.5))
        self.mu_v = nn.Parameter(torch.full((1, 1, dim), 0.5))
        self.W_r  = nn.Parameter(torch.ones(1, 1, dim))
        self.W_k  = nn.Parameter(torch.ones(1, 1, dim))
        self.W_v  = nn.Parameter(torch.ones(1, 1, dim))
        self.W_o  = nn.Linear(dim, dim, bias=False)
        self.wkv  = wkv_cls(dim)

    @staticmethod
    def _shift(x):
        return torch.cat([torch.zeros_like(x[:, :1]), x[:, :-1]], dim=1)

    def forward(self, x):
        p = self._shift(x)
        r = self.W_r * (self.mu_r * x + (1 - self.mu_r) * p)
        k = self.W_k * (self.mu_k * x + (1 - self.mu_k) * p)
        v = self.W_v * (self.mu_v * x + (1 - self.mu_v) * p)
        return self.W_o(torch.sigmoid(r) * self.wkv(k, v))


class HyperMixing(nn.Module):
    def __init__(self, dim, wkv_cls=WKVOperator):
        super().__init__()
        self.mu_r = nn.Parameter(torch.full((1, 1, dim), 0.5))
        self.mu_k = nn.Parameter(torch.full((1, 1, dim), 0.5))
        self.W_r  = nn.Linear(dim, dim, bias=False)
        self.W_k  = nn.Linear(dim, dim, bias=False)
        self.W_h  = nn.Linear(dim, dim, bias=False)
        self.mish = nn.Mish()
        self.wkv  = wkv_cls(dim)

    @staticmethod
    def _shift(x):
        return torch.cat([torch.zeros_like(x[:, :1]), x[:, :-1]], dim=1)

    def forward(self, x):
        p       = self._shift(x)
        r       = self.W_r(self.mu_r * x + (1 - self.mu_r) * p)
        k       = self.W_k(self.mu_k * x + (1 - self.mu_k) * p)
        v_prime = self.wkv(k, x)
        return torch.sigmoid(r) * self.W_h(self.mish(k) * v_prime)


class TinyAttention(nn.Module):
    def __init__(self, dim, num_heads=1):
        super().__init__()
        self.H     = num_heads
        self.D     = dim // num_heads
        self.scale = self.D ** -0.5
        self.qkv   = nn.Linear(dim, 3 * dim, bias=False)
        self.proj  = nn.Linear(dim, dim,     bias=False)

    def forward(self, x):
        B, T, C = x.shape
        qkv  = self.qkv(x).reshape(B, T, 3, self.H, self.D).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        return self.proj((attn.softmax(dim=-1) @ v).transpose(1, 2).reshape(B, T, C))


class TimeMixFormerBlock(nn.Module):
    def __init__(self, dim, num_heads=1, bidirectional=False):
        super().__init__()
        WKV        = BidirectionalWKVOperator if bidirectional else WKVOperator
        self.norm1 = nn.LayerNorm(dim)
        self.tm    = TimeMixing(dim, WKV)
        self.norm2 = nn.LayerNorm(dim)
        self.attn  = TinyAttention(dim, num_heads)

    def forward(self, x):
        xn = self.norm1(x)
        x  = x + self.tm(self.tm(xn))
        x  = x + self.attn(self.norm2(x))
        return x


class HyperMixFormerBlock(nn.Module):
    def __init__(self, dim, num_heads=1, bidirectional=False):
        super().__init__()
        WKV        = BidirectionalWKVOperator if bidirectional else WKVOperator
        self.norm1 = nn.LayerNorm(dim)
        self.hm    = HyperMixing(dim, WKV)
        self.norm2 = nn.LayerNorm(dim)
        self.attn  = TinyAttention(dim, num_heads)

    def forward(self, x):
        xn = self.norm1(x)
        x  = x + self.hm(self.hm(xn))
        x  = x + self.attn(self.norm2(x))
        return x


class SEBlock(nn.Module):
    def __init__(self, dim, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(dim, dim // reduction, bias=False), nn.ReLU(inplace=True),
            nn.Linear(dim // reduction, dim, bias=False), nn.Sigmoid(),
        )
    def forward(self, x):
        s = x.mean(dim=[2, 3])
        return x * self.fc(s).unsqueeze(-1).unsqueeze(-1)


class SinCos2DPositionalEncoding(nn.Module):
    def __init__(self, dim, patch_size):
        super().__init__()
        assert dim % 4 == 0
        P, d    = patch_size, dim // 4
        freq    = 1.0 / (10000 ** (torch.arange(d).float() / d))
        i_idx   = torch.arange(P).repeat_interleave(P)
        j_idx   = torch.arange(P).repeat(P)
        ri      = i_idx.unsqueeze(1) * freq.unsqueeze(0)
        cj      = j_idx.unsqueeze(1) * freq.unsqueeze(0)
        pe      = torch.stack([ri.sin(), ri.cos(), cj.sin(), cj.cos()], dim=-1)
        self.register_buffer('pe', pe.reshape(P * P, dim).unsqueeze(0))
    def forward(self, x): return x + self.pe


class MultiRingCenterAttention(nn.Module):
    def __init__(self, dim, patch_size, num_heads=1):
        super().__init__()
        self.H     = num_heads
        self.D     = dim // num_heads
        self.scale = self.D ** -0.5
        P  = patch_size
        cy, cx          = P // 2, P // 2
        self.center_idx = cy * P + cx
        ring1 = [(cy+di)*P+(cx+dj) for di in (-1,0,1) for dj in (-1,0,1) if not (di==0 and dj==0)]
        ring2 = [i for i in range(P*P) if i != self.center_idx and i not in ring1]
        self.register_buffer('ring1_idx', torch.tensor(ring1))
        self.register_buffer('ring2_idx', torch.tensor(ring2))
        self.norm    = nn.LayerNorm(dim)
        self.q_proj  = nn.Linear(dim, dim,     bias=False)
        self.kv_proj = nn.Linear(dim, 2 * dim, bias=False)
        self.merge   = nn.Linear(2 * dim, dim, bias=False)
        self.proj    = nn.Linear(dim, dim,     bias=False)
        with torch.no_grad():
            self.merge.weight.copy_(0.5 * torch.eye(dim).repeat(1, 2))

    def _ring_attn(self, q, k, v, idx):
        B, n = q.shape[0], idx.shape[0]
        k_r  = k[:, idx].reshape(B, n, self.H, self.D).transpose(1, 2)
        v_r  = v[:, idx].reshape(B, n, self.H, self.D).transpose(1, 2)
        attn = (q @ k_r.transpose(-2, -1)) * self.scale
        return (attn.softmax(dim=-1) @ v_r).transpose(1, 2).reshape(B, 1, self.H * self.D)

    def forward(self, x):
        B, T, C = x.shape
        xn      = self.norm(x)
        center  = xn[:, self.center_idx].unsqueeze(1)
        q       = self.q_proj(center).reshape(B, 1, self.H, self.D).transpose(1, 2)
        k, v    = self.kv_proj(xn).chunk(2, dim=-1)
        out_r1  = self._ring_attn(q, k, v, self.ring1_idx)
        out_r2  = self._ring_attn(q, k, v, self.ring2_idx)
        fused   = self.merge(torch.cat([out_r1, out_r2], dim=-1)).squeeze(1)
        return self.proj(fused) + x[:, self.center_idx]


class MLPHead(nn.Module):
    def __init__(self, in_dim, num_classes, dropout=0.1):
        super().__init__()
        h = in_dim * 4
        self.net = nn.Sequential(
            nn.Linear(in_dim, h), nn.BatchNorm1d(h),
            nn.Dropout(dropout),  nn.ReLU(inplace=True),
            nn.Linear(h, num_classes),
        )
    def forward(self, x): return self.net(x)


class TCFormer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        D  = cfg.hidden_dim
        bi = getattr(cfg, 'use_bidirectional_wkv', False)
        self.stem = nn.Sequential(
            nn.Conv2d(cfg.pca_components, D, kernel_size=cfg.kernel_size,
                      padding=cfg.kernel_size // 2, bias=False),
            nn.BatchNorm2d(D), nn.ReLU(inplace=True),
        )
        self.se      = SEBlock(D) if getattr(cfg, 'use_se_block', True) else None
        self.pos_enc = SinCos2DPositionalEncoding(D, cfg.patch_size) if getattr(cfg, 'use_pos_encoding', True) else None
        self.time_blocks  = nn.ModuleList([TimeMixFormerBlock(D, cfg.num_heads, bi)  for _ in range(cfg.depth)])
        self.hyper_blocks = nn.ModuleList([HyperMixFormerBlock(D, cfg.num_heads, bi) for _ in range(cfg.depth)])
        self.center_attn  = MultiRingCenterAttention(D, cfg.patch_size, cfg.num_heads)
        self.head         = MLPHead(D, cfg.num_classes, cfg.dropout)

    def forward(self, x):
        x = self.stem(x)
        if self.se      is not None: x = self.se(x)
        x = x.flatten(2).transpose(1, 2)
        if self.pos_enc is not None: x = self.pos_enc(x)
        for block in self.time_blocks:  x = block(x)
        for block in self.hyper_blocks: x = block(x)
        return self.head(self.center_attn(x))

## Utils: метрики, evaluate, accelerator

In [7]:
def compute_metrics(preds, trues, num_classes):
    oa = float((preds == trues).mean())
    try:    kappa = float(cohen_kappa_score(trues, preds))
    except: kappa = 0.0
    per_c = [float((preds[trues==c]==trues[trues==c]).mean())
             for c in range(num_classes) if (trues==c).sum() > 0]
    return dict(OA=oa, AA=float(np.mean(per_c)) if per_c else 0.0, Kappa=kappa, per_class=per_c)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    dev = next(model.parameters()).device
    preds, trues = [], []
    for x, y in loader:
        preds.extend(model(x.to(dev)).argmax(1).cpu().tolist())
        trues.extend(y.tolist())
    return np.array(preds), np.array(trues)

def detect_accelerator():
    if torch.backends.mps.is_available():  return 'mps', torch.device('mps')
    if torch.cuda.is_available():           return 'gpu', torch.device('cuda')
    return 'cpu', torch.device('cpu')

def get_run_dir(name, results_root='results'):
    base = Path(results_root)
    v = 1
    while (base / f'{name}_v{v}').exists(): v += 1
    d = base / f'{name}_v{v}'
    d.mkdir(parents=True)
    return d

def get_seed_dir(run_dir, seed):
    d = run_dir / 'seeds' / f'seed_{seed}'
    d.mkdir(parents=True)
    return d

## Lightning wrapper

In [8]:
class EpochLogger(L.Callback):
    def __init__(self, log_every=1): self.log_every = log_every
    def on_train_epoch_start(self, trainer, pl_module):
        ep = trainer.current_epoch + 1
        print(f'  Ep {ep:3d}/{trainer.max_epochs} ({ep/trainer.max_epochs:3.0%}) ...', end='', flush=True)
    def on_validation_epoch_end(self, trainer, pl_module):
        epoch = trainer.current_epoch + 1
        if epoch % self.log_every != 0 and epoch != 1:
            print()
            return
        print(f'\r  Ep {epoch:3d}/{trainer.max_epochs} ({epoch/trainer.max_epochs:3.0%})'
              f'  train: loss={pl_module._log_train_loss:.4f}  acc={pl_module._log_train_acc:.1%}'
              f'  │  val: loss={pl_module._log_val_loss:.4f}'
              f'  OA={pl_module._log_val_oa:.1%}  AA={pl_module._log_val_aa:.1%}'
              f'  κ={pl_module._log_val_kappa:.4f}')


class TCFormerLit(L.LightningModule):
    def __init__(self, cfg):
        super().__init__()
        self.cfg       = cfg
        self.model     = TCFormer(cfg)
        self.criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
        self._val_preds = []; self._val_trues  = []; self._val_losses   = []
        self._train_losses = []; self._train_accs = []
        self._log_train_loss = self._log_train_acc = float('nan')
        self._log_val_loss = self._log_val_oa = self._log_val_aa = self._log_val_kappa = float('nan')

    def forward(self, x): return self.model(x)

    def training_step(self, batch, _):
        x, y   = batch
        logits = self(x)
        loss   = self.criterion(logits, y)
        acc    = (logits.argmax(1) == y).float().mean()
        self.log('train_loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('train_acc',  acc,  prog_bar=True, on_step=False, on_epoch=True)
        self._train_losses.append(loss.item()); self._train_accs.append(acc.item())
        return loss

    def on_train_epoch_end(self):
        if self._train_losses:
            self._log_train_loss = float(np.mean(self._train_losses))
            self._log_train_acc  = float(np.mean(self._train_accs))
            self._train_losses.clear(); self._train_accs.clear()

    def validation_step(self, batch, _):
        x, y = batch
        logits = self(x)
        self._val_preds.append(logits.argmax(1).cpu())
        self._val_trues.append(y.cpu())
        self._val_losses.append(self.criterion(logits, y).item())

    def on_validation_epoch_end(self):
        if not self._val_preds: return
        preds = torch.cat(self._val_preds).numpy()
        trues = torch.cat(self._val_trues).numpy()
        m = compute_metrics(preds, trues, self.cfg.num_classes)
        self.log('val_OA', m['OA'], prog_bar=True)
        self.log('val_AA', m['AA'], prog_bar=False)
        self.log('val_kappa', m['Kappa'], prog_bar=False)
        self.log('val_loss', float(np.mean(self._val_losses)), prog_bar=False)
        self._log_val_oa = m['OA']; self._log_val_aa = m['AA']
        self._log_val_kappa = m['Kappa']; self._log_val_loss = float(np.mean(self._val_losses))
        self._val_preds.clear(); self._val_trues.clear(); self._val_losses.clear()

    def configure_optimizers(self):
        opt   = torch.optim.Adam(self.parameters(), lr=self.cfg.lr, weight_decay=self.cfg.weight_decay)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.cfg.epochs, eta_min=self.cfg.lr * 0.1)
        return [opt], [{'scheduler': sched, 'interval': 'epoch'}]

## Навчання

In [9]:
cfg           = CONFIGS[DATASET]
cfg.data_path = DATA_PATH
if PAPER_MODE: cfg.num_val_per_class = 0

suffix  = '_paper' if PAPER_MODE else ''
run_dir = get_run_dir(f'improved{suffix}_{DATASET}', results_root=RESULTS)
print(f'Run directory: {run_dir}')
with open(run_dir / 'config.json', 'w') as f:
    json.dump(dataclasses.asdict(cfg), f, indent=2, ensure_ascii=False)

hsi, labels = load_dataset(cfg.dataset, cfg.data_path)
hsi_pca, _  = apply_pca(hsi, cfg.pca_components)
if cfg.remove_pca_outliers:
    labels = remove_pca_outliers(labels, hsi_pca, cfg.outlier_std)

Run directory: results/improved_IP_v3
[IP] HSI (145, 145, 200)  Labels (145, 145)  Labeled px: 10249
PCA 200 -> 150  explained variance: 1.000
Outlier removal (>2.5σ): 305 px removed (3.0%)  |  labeled: 10249 -> 9944


In [10]:
def train_one_seed(seed, run_dir, cfg, hsi_pca, labels, paper_mode=True, log_every=5):
    L.seed_everything(seed, workers=True)
    cfg.seed   = seed
    seed_dir   = get_seed_dir(run_dir, seed)
    print(f'\n── Seed {seed} → {seed_dir} ──')

    train_loader, val_loader, test_loader = get_dataloaders(hsi_pca, labels, cfg)
    if paper_mode:
        val_loader = test_loader
        print('paper_mode: test set used as eval')

    lit             = TCFormerLit(cfg)
    accelerator, _  = detect_accelerator()
    checkpoint_cb   = ModelCheckpoint(
        dirpath=seed_dir / 'checkpoints', monitor='val_OA', mode='max',
        save_top_k=1, filename='best-{epoch:03d}-{val_OA:.4f}', verbose=False,
    )
    trainer = L.Trainer(
        max_epochs=cfg.epochs, accelerator=accelerator, devices=1,
        callbacks=[checkpoint_cb,
                   EarlyStopping(monitor='val_OA', mode='max', patience=cfg.patience, verbose=False),
                   EpochLogger(log_every)],
        logger=CSVLogger(save_dir=str(seed_dir), name='', version=''),
        log_every_n_steps=1, enable_progress_bar=False,
        enable_model_summary=False, num_sanity_val_steps=0,
    )
    trainer.fit(lit, train_loader, val_loader)

    best_oa = float(checkpoint_cb.best_model_score) if checkpoint_cb.best_model_score else 0.0
    print(f'Best val_OA={best_oa:.4f}')

    ckpt = torch.load(checkpoint_cb.best_model_path, map_location='cpu', weights_only=False)
    lit.load_state_dict(ckpt['state_dict'])
    _, dev = detect_accelerator()
    lit.model.to(dev)

    preds, trues = evaluate(lit.model, test_loader)
    m = compute_metrics(preds, trues, cfg.num_classes)
    print(f'[seed={seed}] OA={m["OA"]*100:.2f}%  AA={m["AA"]*100:.2f}%  Kappa={m["Kappa"]:.4f}')

    with open(seed_dir / 'test_metrics.json', 'w') as f:
        json.dump({'seed': seed, 'dataset': DATASET,
                   'OA': round(m['OA']*100,2), 'AA': round(m['AA']*100,2),
                   'Kappa': round(m['Kappa'],4), 'per_class': m['per_class']}, f, indent=2)
    return m


seed_results = []
for seed in SEEDS:
    seed_results.append(train_one_seed(seed, run_dir, cfg, hsi_pca, labels, True, LOG_EVERY))

Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.



── Seed 42 → results/improved_IP_v3/seeds/seed_42 ──
Split — Train: 160  Val: 80  Test: 9704
paper_mode: test set used as eval
  Ep   1/100 ( 1%) ...


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

## Агрегація результатів

In [ ]:
oas    = [r['OA']    for r in seed_results]
aas    = [r['AA']    for r in seed_results]
kappas = [r['Kappa'] for r in seed_results]

agg = {
    'dataset': DATASET, 'seeds': SEEDS,
    'OA':    f'{round(np.mean(oas)*100,2)} ± {round(np.std(oas)*100,2)}',
    'AA':    f'{round(np.mean(aas)*100,2)} ± {round(np.std(aas)*100,2)}',
    'Kappa': f'{round(np.mean(kappas),4)} ± {round(np.std(kappas),4)}',
}
with open(run_dir / 'test_metrics.json', 'w') as f:
    json.dump(agg, f, indent=2, ensure_ascii=False)

print(f'\n=== Aggregated [{DATASET}] ({len(SEEDS)} seed(s)) ===')
print(f'OA    : {agg["OA"]}')
print(f'AA    : {agg["AA"]}')
print(f'Kappa : {agg["Kappa"]}')
print(f'Results: {run_dir}')